In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('encoder.py'), '..')))
from encoder import EncoderBuilder, Sampling

import numpy as np
import tensorflow as tf
from keras.models import load_model, Model
from keras.layers import Dense, Input, Dropout, BatchNormalization
from keras.optimizers import Adam
import h5py
from sklearn.model_selection import train_test_split

2025-08-17 14:37:18.732393: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-17 14:37:18.779166: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-08-17 14:37:19.986559: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
encoder_model = load_model("/home/chua/projects/Autoencoder Notes/Interpretable Sound Generation/initial_encoder_vae_log_mel_spec.keras",
    custom_objects={
        "Sampling" : Sampling
    })

I0000 00:00:1755412641.358377   14417 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9711 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [3]:
with h5py.File('Dataset/log_mel_spec_data_dataset.h5', 'r') as h5f:
    log_mel_spec_data_train = h5f['train'][:]
    log_melspec_data_labels = h5f['label'][:]

In [4]:
mean_vector, log_variance_vector, latent_vector_representation = encoder_model.predict(log_mel_spec_data_train)

2025-08-17 14:37:24.737617: I external/local_xla/xla/service/service.cc:163] XLA service 0x7d1f4001c350 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-08-17 14:37:24.737649: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2025-08-17 14:37:24.746163: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-08-17 14:37:24.784276: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 90300


 25/938 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step

I0000 00:00:1755412647.221093   14497 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


938/938 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step


In [5]:
print(type(mean_vector), mean_vector.shape)
print(type(log_melspec_data_labels), log_melspec_data_labels.shape)

<class 'numpy.ndarray'> (30000, 128)
<class 'numpy.ndarray'> (30000,)


In [6]:
train = mean_vector
label = log_melspec_data_labels

In [7]:
input = Input(shape=(128,))
x = Dense(128, activation='relu')(input)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
output = Dense(10, activation='softmax')(x)
model = Model(inputs=input, outputs=output)

In [8]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    train, label, test_size=0.2, random_state=42, stratify=label
)

In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,186 (102.29 KB)

 Trainable params: 25,802 (100.79 KB)

 Non-trainable params: 384 (1.50 KB)

In [11]:
history = model.fit(X_train, y_train, validation_data=[X_test, y_test], epochs=20, batch_size=64)

Epoch 1/20


2025-08-17 14:37:38.011543: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-08-17 14:37:38.011598: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-08-17 14:37:38.011627: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-08-17 14:37:38.011638: I external/l

360/375 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7160 - loss: 0.9398

2025-08-17 14:37:45.149940: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-08-17 14:37:45.149995: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-08-17 14:37:45.838363: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_61', 20 bytes spill stores, 20 bytes spill loads

2025-08-17 14:37:46.183534: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : R

375/375 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.7225 - loss: 0.9189 - val_accuracy: 0.9772 - val_loss: 0.1037
Epoch 2/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9586 - loss: 0.1349 - val_accuracy: 0.9862 - val_loss: 0.0475
Epoch 3/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9740 - loss: 0.0879 - val_accuracy: 0.9900 - val_loss: 0.0347
Epoch 4/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9811 - loss: 0.0615 - val_accuracy: 0.9907 - val_loss: 0.0309
Epoch 5/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9827 - loss: 0.0537 - val_accuracy: 0.9907 - val_loss: 0.0289
Epoch 6/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9872 - loss: 0.0406 - val_accuracy: 0.9913 - val_loss: 0.0275
Epoch 7/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9849 - loss: 0.0472 - val_accuracy: 0.9933 - val_loss: 0.0242
Epoch 8/20
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9903 - loss: 0.0340 - val_accuracy: 0.9920 - va

In [12]:
model.save("log_mel_mu_classifier.keras")

In [ ]:
X_test.shape

(6000, 128)

In [14]:

y_test.shape

(6000,)

In [15]:
np.save("mean_vector_test_x_set.npy", X_test)
np.save("mean_vector_test_y_set.npy", y_test)